# Higgs-Audio v2 (3B) ile anlatım üretimi

Okuma Kütüphanesi için stüdyo kalitesinde anlatım sesi üretir.

**Model:** `bosonai/higgs-audio-v2-generation-3B-base`

**Önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**

Referans kayıt **isteğe bağlıdır**: verilmezse model kendi
doğal anlatım sesini üretir.

Hücreleri sırayla çalıştırın. Son hücre `audio-<docid>.zip` indirir;
onu sitedeki `assets/audio/<docid>/` klasörüne açın.


In [ ]:
#@title 1 · GPU ve ortam denetimi
import subprocess, sys, torch
g = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                    '--format=csv,noheader'],
                   capture_output=True, text=True).stdout.strip()
print('GPU      :', g or 'YOK — Çalışma zamanı türünü T4 GPU yapın')
print('torch    :', torch.__version__)
print('python   :', sys.version.split()[0])
print('cuda     :', torch.version.cuda)


In [ ]:
#@title 2 · Metinleri çek ve belgeyi seç
REPO = 'https://github.com/adzetto/reading-library'
import os, json

if not os.path.exists('reading-library'):
    !git clone -q --depth 1 $REPO reading-library
else:
    !cd reading-library && git pull -q

NARR = 'reading-library/narration'
index = json.load(open(f'{NARR}/INDEX.json', encoding='utf-8'))
index.sort(key=lambda r: r['chars'])
print(f"{'belge':22s}{'parça':>7s}{'karakter':>10s}{'~dk ses':>9s}")
print('-'*50)
for r in index:
    print(f"{r['id']:22s}{r['items']:7d}{r['chars']:10d}{r['minutes_est']:9d}")
print('\nEn kısadan başlayın: aesop-fables (~5 dk üretim)')


In [ ]:
#@title 3 · Belge seçimi
DOC_ID = 'aesop-fables'  #@param ['aesop-fables','selfish-giant','tell-tale-heart','alice-rabbit-hole','happy-prince','ugly-duckling','net-feasibility','jadr-2022','doc-b89f','doc-net-tr']
#@markdown Uzun belgeleri bölmek için parça aralığı (0,0 = tümü)
PARCA_BAS = 0  #@param {type:'integer'}
PARCA_SON = 0  #@param {type:'integer'}

doc = json.load(open(f'{NARR}/{DOC_ID}.json', encoding='utf-8'))
items = doc['items']
if PARCA_SON: items = items[PARCA_BAS:PARCA_SON]
elif PARCA_BAS: items = items[PARCA_BAS:]

print(doc['title'][:80])
print(f'{len(items)} / {len(doc["items"])} parça seçildi')
print('ton önerisi :', doc['voice_hint'])
print('en uzun     :', max(x['chars'] for x in items), 'karakter')
print('\nörnek:', items[0]['text'][:180])


## Kurulum

İlk çalıştırmada ~7 GB indirir.


In [ ]:
#@title 4 · Higgs-Audio v2 kurulumu
!pip -q install --upgrade 'transformers>=4.45' accelerate torchaudio soundfile 2>/dev/null | tail -1
!pip -q install git+https://github.com/boson-ai/higgs-audio.git 2>/dev/null | tail -1

import torch
MODEL_LABEL = 'bosonai/higgs-audio-v2-generation-3B-base'
TOKENIZER_ID = 'bosonai/higgs-audio-v2-tokenizer'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 24000

from boson_multimodal.serve.serve_engine import HiggsAudioServeEngine
engine = HiggsAudioServeEngine(MODEL_LABEL, TOKENIZER_ID, device=DEV)
print('model hazır ·', DEV)


In [ ]:
#@title 5 · Çıktı klasörü
#@markdown Drive bağlarsanız oturum kopsa da üretilen parçalar durur.
USE_DRIVE = True  #@param {type:'boolean'}
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = f'/content/drive/MyDrive/reading-library-audio/{DOC_ID}'
else:
    OUT = f'/content/out/{DOC_ID}'
os.makedirs(OUT, exist_ok=True)
done = {f.rsplit('.',1)[0] for f in os.listdir(OUT) if f.endswith('.wav')}
print('çıktı klasörü   :', OUT)
print('zaten üretilmiş :', len(done), 'parça')
print('üretilecek      :', len([i for i in items if i['id'] not in done]))


In [ ]:
#@title 6 · Ses ve üslup
REF_WAV = ''  #@param {type:'string'}
VOICE_LABEL = 'higgs-v2 · doğal anlatım'  #@param {type:'string'}
TEMPERATURE = 0.3  #@param {type:'number'}

AKADEMIK = ('A single narrator reads an academic text aloud: calm, '
            'clear, unhurried, neutral accent.')
OYKU     = ('A single narrator reads a story aloud: warm, expressive, '
            'gently paced, natural storytelling rhythm.')
ton = OYKU if 'story' in doc['voice_hint'] else AKADEMIK

SYSTEM = ('Generate audio following instruction.\n\n'
          '<|scene_desc_start|>\n'
          'Audio is recorded in a quiet studio. ' + ton +
          ' No music, no background noise.\n'
          '<|scene_desc_end|>')
print(SYSTEM)


In [ ]:
#@title 7 · Üretim  (kesilirse aynı hücreyi tekrar çalıştırın)
import time, os, soundfile as sf, traceback
from boson_multimodal.data_types import ChatMLSample, Message, AudioContent

def mesajlar(text):
    m = [Message(role='system', content=SYSTEM)]
    if REF_WAV:
        m.append(Message(role='user', content='Reference voice.'))
        m.append(Message(role='assistant',
                         content=AudioContent(audio_url=REF_WAV)))
    m.append(Message(role='user', content=text))
    return ChatMLSample(messages=m)

todo = [i for i in items if i['id'] not in done]
print(f'{len(todo)} parça üretilecek\n')
t0, hata = time.time(), 0

for n, it in enumerate(todo, 1):
    try:
        out = engine.generate(chat_ml_sample=mesajlar(it['text']),
                              max_new_tokens=2048,
                              temperature=TEMPERATURE, top_p=0.95, top_k=50)
        sf.write(f"{OUT}/{it['id']}.wav", out.audio, out.sampling_rate)
        SR = out.sampling_rate
    except Exception as e:
        hata += 1
        print(f"  ! {it['id']}: {type(e).__name__}: {e}")
        if hata <= 1: traceback.print_exc()
        continue
    if n % 5 == 0 or n == len(todo):
        hz = n / (time.time() - t0)
        print(f'{n}/{len(todo)} · {hz*60:.1f} parça/dk · '
              f'~{(len(todo)-n)/max(hz,1e-9)/60:.0f} dk kaldı')

print(f'\nbitti · {hata} hata')
from IPython.display import Audio, display
ilk = f"{OUT}/{items[0]['id']}.wav"
if os.path.exists(ilk):
    print('\nörnek:', items[0]['text'][:120]); display(Audio(ilk))


In [ ]:
#@title 8 · WAV → OPUS, manifest ve indirme
#@markdown Opus çok daha küçük; konuşma için kalite yeterli.
TO_OPUS = True   #@param {type:'boolean'}
BITRATE = '48k'  #@param {type:'string'}

import os, json, glob, subprocess, wave, contextlib
!apt-get -qq install -y ffmpeg zip > /dev/null 2>&1

fmt  = 'opus' if TO_OPUS else 'wav'
pack = f'/content/pack/{DOC_ID}'
os.makedirs(pack, exist_ok=True)

entries, total, missing = [], 0.0, 0
for it in doc['items']:
    src = f"{OUT}/{it['id']}.wav"
    if not os.path.exists(src):
        missing += 1
        continue
    with contextlib.closing(wave.open(src)) as w:
        dur = round(w.getnframes() / float(w.getframerate()), 2)
    dst = f"{pack}/{it['id']}.{fmt}"
    if TO_OPUS:
        subprocess.run(['ffmpeg','-y','-loglevel','error','-i',src,
                        '-c:a','libopus','-b:a',BITRATE,'-ac','1',dst], check=True)
    else:
        subprocess.run(['cp', src, dst], check=True)
    entries.append({'id': it['id'], 'node': it['node'],
                    'part': it['part'], 'dur': dur})
    total += dur

manifest = {'id': DOC_ID, 'voice': VOICE_LABEL, 'model': MODEL_LABEL,
            'format': fmt, 'sr': SR, 'items': entries}
with open(f'{pack}/manifest.js', 'w', encoding='utf-8') as f:
    f.write('window.AUDIO = window.AUDIO || {};\n')
    f.write(f'window.AUDIO["{DOC_ID}"] = ')
    json.dump(manifest, f, ensure_ascii=False)
    f.write(';\n')

mb = sum(os.path.getsize(p) for p in glob.glob(f'{pack}/*')) / 1048576
print(f'{len(entries)} parça · {total/60:.1f} dakika ses · {mb:.1f} MB')
if missing:
    print(f'UYARI: {missing} parça eksik — 7. hücreyi tekrar çalıştırın')

!cd /content/pack && zip -qr audio-$DOC_ID.zip $DOC_ID
from google.colab import files
files.download(f'/content/pack/audio-{DOC_ID}.zip')


## Siteye koyma

```
1. audio-<docid>.zip dosyasını açın
2. çıkan <docid>/ klasörünü  assets/audio/  altına koyun
3. assets/audio/index.js içine id'yi ekleyin:
     window.AUDIO_INDEX = ['aesop-fables'];
```

Site açıldığında seslendirme panelinde **Stüdyo ses** rozeti belirir.
Paragraf vurgusu ve iki dilli transkript kendiliğinden hizalanır.
